In [ ]:
from complete_pipeline import CompletePipeline

pipeline = CompletePipeline(
    csv_path='../refactored_semantic/dbbe_full.csv',
    scratch_dir='/scratch/gent/vo/000/gvo00042/vsc48660',
    output_dir='./results',
    num_perms=[128],
    shingle_sizes=[3, 4],
    thresholds=[0.75, 0.8, 0.85],
    poem_thresholds=[0.2, 0.3, 0.4, 0.5],
    n_jobs=-1
)

pipeline.run_full_pipeline()

## Print sample results

In [ ]:
import pandas as pd
import numpy as np
df=pd.read_csv('results/verse_clusters.csv')
df=df.dropna()
def show_example_verse_clusters(df: pd.DataFrame, n_clusters: int = 3):
    clusters = (
        df['predicted_verse_cluster']
        .dropna()
        .astype(int)
        .unique()
    )
    cluster_sizes = (
    df.groupby("predicted_verse_cluster")
      .size()
      .rename("count")
    )

    clusters_with_multiple = cluster_sizes[cluster_sizes > 1].index.tolist()
    example_clusters = np.random.choice(clusters_with_multiple, size=min(n_clusters, len(clusters)), replace=False)
    
    for cid in example_clusters:
        print(f"Cluster {cid}")
        
        subset = df[df['predicted_verse_cluster'] == cid]
        
        for _, row in subset.iterrows():
            print(f"Verse: {row['verse']}")
        
        print("\n")
show_example_verse_clusters(df, n_clusters=10)

In [ ]:
import pandas as pd

poems_df = pd.read_csv("results/poem_clusters.csv")
poems_df["idoriginal_poem"] = poems_df["idoriginal_poem"].astype(str)

cluster_sizes = poems_df.groupby("predicted_poem_cluster").size()
multi_poem_clusters = cluster_sizes[cluster_sizes > 1].index.tolist()
print(f"Found {len(multi_poem_clusters)} multi-poem clusters.")

clusters_to_show = multi_poem_clusters[:3]
poems_to_show = poems_df[poems_df["predicted_poem_cluster"].isin(clusters_to_show)]

cluster_to_poems = (
    poems_to_show.groupby("predicted_poem_cluster")["idoriginal_poem"]
    .apply(list)
    .to_dict()
)

verse_dict = {}  
chunksize = 100_000

for chunk in pd.read_csv("../concatenated.csv", chunksize=chunksize):
    chunk["idoriginal_poem"] = chunk["idoriginal_poem"].astype(str)
    chunk = chunk[chunk["idoriginal_poem"].isin(poems_to_show["idoriginal_poem"])]
    
    for poem_id, group in chunk.groupby("idoriginal_poem"):
        verses = group.sort_values("order")["verse"].tolist()
        verses = [str(v) for v in verses if pd.notna(v)]
        if poem_id in verse_dict:
            verse_dict[poem_id].extend(verses)
        else:
            verse_dict[poem_id] = verses

for cl in clusters_to_show:
    print("\n" + "="*80)
    print(f"CLUSTER {cl}")
    print("="*80)
    
    for poem_id in cluster_to_poems[cl]:
        poem_text = "\n".join(verse_dict.get(poem_id, []))
        print(f"\n--- Poem {poem_id} ---\n")
        print(poem_text)
        print("\n" + "-"*40)
